# SDL — browse & fetch (retrieve published files for postprocessing)

Download files that were published to the **SDL** back into HySEALab, e.g. to **postprocess** a
simulation. The SDL is a publication/archive store (not a mounted filesystem), so the flow is:
**find the experiment → list its files → download the ones you want → process them locally.**

**Flow:** *Setup → Login → (optional) list your experiments → enter the experiment id → load version →
list files → select → download.* Tip: it is often easiest to **browse in the SDL web portal** (maps,
previews) and just **paste the experiment id** here.

> Downloads go to a local folder in your HySEALab home (default `~/sdl_downloads/exp<id>_v<ver>/`),
> preserving the file paths. You can then open/postprocess them with your usual notebooks.


## 1. Setup & dependencies

In [ ]:
# --- Setup: load the companion library sdl_lib.py and check dependencies -----
import os, sys, importlib, importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Locate sdl_lib.py: it lives either in a sibling 01_sdl_publish/ folder (repo
# layout) or directly in the parent folder (server ~/SDL layout). Search both.
_cands = [os.path.join(os.getcwd(), "..", "01_sdl_publish"),
          os.path.join(os.getcwd(), ".."),
          os.getcwd(),
          os.path.expanduser("~/HySEALab/Notebooks/01_sdl_publish"),  # HySEALab menu: shared lib
          os.path.expanduser("~/HySEALab/Notebooks"),                 # shared (next to map viewer)
          os.path.expanduser("~/SDL")]                                # personal working copy
LIB_DIR = next((os.path.abspath(d) for d in _cands
                if os.path.exists(os.path.join(d, "sdl_lib.py"))), None)
if LIB_DIR is None:
    raise FileNotFoundError("sdl_lib.py not found in ../01_sdl_publish, .. or .")
sys.path.insert(0, LIB_DIR)

REQUIRED = {
    "requests": "requests", "requests_toolbelt": "requests-toolbelt",
    "rasterio": "rasterio", "cmocean": "cmocean",
    "matplotlib": "matplotlib", "numpy": "numpy", "tqdm": "tqdm",
    "yaml": "pyyaml", "jsonschema": "jsonschema",
}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

_dep_out = widgets.Output()
def _install_missing(_):
    import subprocess
    with _dep_out:
        clear_output()
        print("Installing into the user site:", " ".join(_missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", *_missing])
        print("Done. Restart the kernel and re-run this cell.")
_btn_install = widgets.Button(description="Install missing deps", button_style="warning")
_btn_install.on_click(_install_missing)

if _missing:
    print("Missing dependencies:", ", ".join(_missing))
    display(_btn_install, _dep_out)
else:
    print("All dependencies available.")

import sdl_lib
importlib.reload(sdl_lib)
print("sdl_lib loaded from:", LIB_DIR)

# Shared state for this notebook (login token lives only in memory).
S = {"base_url": "https://sdl-dev.hpc.cineca.it", "token": None, "headers": None}
print("Setup ready.")


## 2. Login

Use the **same server** where the experiment lives (dev/prod).

In [ ]:
# --- Login: authenticate once; the token is reused by the publisher ----------
w_env = widgets.Dropdown(
    options=[("Development (sdl-dev.hpc.cineca.it)", "https://sdl-dev.hpc.cineca.it"),
             ("Production  (sdl.hpc.cineca.it)", "https://sdl.hpc.cineca.it")],
    value="https://sdl-dev.hpc.cineca.it", description="Server:",
    layout=widgets.Layout(width="440px"))
w_user = widgets.Text(description="Username:", placeholder="your SDL e-mail",
                      layout=widgets.Layout(width="440px"))
w_pass = widgets.Password(description="Password:", layout=widgets.Layout(width="440px"))
w_login = widgets.Button(description="Login", button_style="primary")
out_login = widgets.Output()

def _do_login(_):
    with out_login:
        clear_output()
        S["base_url"] = w_env.value
        u = w_user.value.strip()
        if not u or not w_pass.value:
            print("Enter username and password."); return
        try:
            tok = sdl_lib.get_auth_token(S["base_url"], u, w_pass.value)
        except Exception as e:
            print("Login failed:", e); return
        if not tok:
            print("Login failed: no token returned."); return
        S["token"] = tok
        S["headers"] = sdl_lib.auth_headers(tok)
        print("Login OK - token stored. Server:", S["base_url"])
w_login.on_click(_do_login)
display(widgets.VBox([w_env, w_user, w_pass, w_login, out_login]))


## 3. (Optional) List your experiments

If you do not have the id handy, list the experiments you can access and copy one id.

In [ ]:
# --- (Optional) List the experiments you can access -------------------------
le_btn = widgets.Button(description="List my experiments", button_style="info")
le_q = widgets.Text(description="Filter:", placeholder="title substring; empty = all",
                    layout=widgets.Layout(width="460px"))
le_out = widgets.Output()

def _le(_):
    with le_out:
        clear_output()
        if not S.get("token"):
            print("Login first (section 2)."); return
        exps = sdl_lib.list_experiments(S["base_url"], S["headers"])
        q = le_q.value.strip().lower()
        rows = [(e.get("id"), (e.get("name") or e.get("title") or "")) for e in exps]
        if q:
            rows = [(i, t) for i, t in rows if q in str(t).lower()]
        if not rows:
            print("No experiments match (or no access)."); return
        print(f"{len(rows)} experiment(s) - copy an id into the field below:\n")
        for i, t in rows:
            print(f"  id={str(i):>6}  {t}")
le_btn.on_click(_le)
display(widgets.VBox([widgets.HBox([le_q, le_btn]), le_out]))


## 4. Browse files & download

Enter the **experiment id** → **Load versions** → pick a **version** → **List files** → select the
files (Ctrl/Cmd-click) → set **Save to** → **Download selected**. Large files stream with a progress
bar.

In [ ]:
# --- Browse an experiment's files and download them into HySEALab ------------
bf_eid = widgets.IntText(description="Experiment id:", value=0, layout=widgets.Layout(width="240px"))
bf_loadv = widgets.Button(description="Load versions", button_style="info")
bf_ver = widgets.Dropdown(description="Version:", options=[], layout=widgets.Layout(width="220px"))
bf_list = widgets.Button(description="List files", button_style="info")
bf_files = widgets.SelectMultiple(options=[], description="Files:", rows=10,
                                  layout=widgets.Layout(width="780px"))
bf_dest = widgets.Text(description="Save to:", layout=widgets.Layout(width="780px"),
                       value=os.path.expanduser("~/sdl_downloads"))
bf_dl = widgets.Button(description="Download selected", button_style="warning")
bf_out = widgets.Output()

def _human(n):
    n = float(n or 0)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024:
            return (f"{n:.0f} {u}" if u == "B" else f"{n:.1f} {u}")
        n /= 1024
    return f"{n:.1f} PB"

def _bf_loadv(_):
    with bf_out:
        clear_output()
        if not S.get("token"):
            print("Login first (section 2)."); return
        if bf_eid.value <= 0:
            print("Enter an experiment id (from the SDL portal, or the list above)."); return
        vs = sdl_lib.list_versions(S["base_url"], S["headers"], bf_eid.value)
        opts = [v.get("version") for v in vs if v.get("version")]
        bf_ver.options = opts
        print(f"Experiment {bf_eid.value}: versions = {opts or '(none / not found / no access)'}")
bf_loadv.on_click(_bf_loadv)

def _bf_list(_):
    with bf_out:
        clear_output()
        if not S.get("token"):
            print("Login first."); return
        if not bf_ver.value:
            print("Load versions and pick one first."); return
        files = sdl_lib.list_files(S["base_url"], S["headers"], bf_eid.value, bf_ver.value)
        files = [f for f in files if (f.get("Size") or 0) > 0]   # skip folder markers
        if not files:
            print("No downloadable files in this version."); return
        bf_files.options = [(f"{f['Key']}  ({_human(f.get('Size'))})", f["Key"])
                            for f in sorted(files, key=lambda x: x["Key"])]
        total = sum(f.get("Size") or 0 for f in files)
        print(f"{len(files)} file(s), {_human(total)} total. Ctrl/Cmd-click to select, "
              f"set 'Save to', then 'Download selected'.")
bf_list.on_click(_bf_list)

def _bf_dl(_):
    with bf_out:
        clear_output()
        if not S.get("token"):
            print("Login first."); return
        keys = list(bf_files.value)
        if not keys:
            print("Select at least one file."); return
        root = os.path.expanduser(bf_dest.value.strip() or "~/sdl_downloads")
        base_dir = os.path.join(root, f"exp{bf_eid.value}_v{bf_ver.value}")
        print(f"Downloading {len(keys)} file(s) to {base_dir} ...")
        for k in keys:
            dest = os.path.join(base_dir, k.lstrip("/"))
            try:
                sdl_lib.download_file(S["base_url"], S["headers"],
                                      bf_eid.value, bf_ver.value, k, dest, log=print)
            except Exception as e:
                print(f"  failed {k}: {type(e).__name__} {e}")
        print("\nDone. Files are under:", base_dir)
bf_dl.on_click(_bf_dl)

display(widgets.VBox([widgets.HBox([bf_eid, bf_loadv, bf_ver, bf_list]),
                      bf_files, bf_dest, bf_dl, bf_out]))


## Notes

- Backed by `sdl_lib.list_files` and `sdl_lib.download_file` (endpoint
  `GET /api/experiments/{id}/versions/{v}/file/stream?path=<key>`); experiment/version listing via
  `list_experiments` / `list_versions`.
- **Private** experiments need login (as here); access depends on your SDL account.
- For your *own* fresh runs the outputs usually already live in `/mnt/scratch` — this tool is for
  retrieving **published/archived** data (yours or shared) back for analysis.
